In [11]:
from google.colab import drive
import os,sys
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
sys.path.append('/content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT')

In [13]:
%%writefile /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re
import pickle

Overwriting /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [14]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

__all__ =[
    'compute_mean_attention_dist',
    'plot_attention_distance'
]

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [15]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def init_t_xy(end_x: int, end_y: int):
  t = torch.arange(end_x * end_y,dtype=torch.float32)
  t_x = (t % end_x).float()
  t_y = torch.div(t, end_x, rounding_mode="floor").float()

  return t_x, t_y

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [16]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def compute_distance_matrix(length):
  t_x, t_y = init_t_xy(length, length)
  coords = torch.stack([t_x, t_y], dim=1)

  diff = coords[:, None, :] - coords[None, :, :]
  dist = torch.sqrt((diff ** 2).sum(dim=-1))
  return dist

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [17]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def compute_mean_attention_dist(scores, patch_size=1):
  # scores: (n_layers, B, heads, 1+N, 1+N), softmax 이후 값
  scores = scores[..., 1:, 1:]                        # (n_layers, B, heads, N, N)
  #scores = scores / scores.sum(dim=-1, keepdim=True)   재정규화는 건너뛰기

  num_patches = scores.shape[-1]
  length = math.isqrt(num_patches)
  assert length ** 2 == num_patches, "Num patches is not perfect square"

  dist = compute_distance_matrix(length).to(scores.device) * patch_size
  dist = dist[None, None, None]                       # (1, 1, 1, N, N)

  mean_dist = (scores * dist).sum(dim=-1)             # (n_layers, B, heads, N)
  return mean_dist.mean(dim=-1).mean(dim=1)           # (n_layers, heads)

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [18]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def plot_attention_distance(
    dist,
    title="ViT",
    ax=None,
    legend_heads=3
):
    dist = (
        dist.detach().cpu().numpy()
        if hasattr(dist, "detach")
        else np.asarray(dist)
    )

    n_layers, n_heads = dist.shape

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    cmap = plt.get_cmap("tab20" if n_heads > 10 else "tab10")
    layers = np.arange(1, n_layers + 1)

    for h in range(n_heads):
        label = f"Head {h + 1}" if h < legend_heads else None

        ax.plot(
            layers,
            dist[:, h],
            marker="o",
            markersize=3,
            linewidth=1,
            color=cmap(h % cmap.N),
            label=label
        )

    ax.set_title(title)
    ax.set_xlabel("Network depth (layer)")
    ax.set_ylabel("Mean attention distance (pixels)")
    ax.set_xlim(0, n_layers + 1)
    ax.set_ylim(bottom=0)

    if legend_heads > 0:
        ax.legend(loc="best")

    return ax

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


# Attention Maps

In [19]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def attention_rollout(scores):
    n_layers, B, H, T, _ = scores.shape
    eye=torch.eye(T,device=scores.device)
    rollout=eye.expand(B,-1,-1)

    for attn in scores:
        attn = attn.mean(dim=1)
        attn = 0.5*attn + 0.5*eye
        rollout = attn @ rollout
    return rollout #(B,T,T)

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


In [20]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py

def plot_attention_map(images, targets, scores, logits,
                       class_names=None, mean=None, std=None,
                       max_images=4, mode="mask"):
  # images : (B, C, H, W)
  # targets: (B,)
  # scores : (n_layers, B, heads, N+1, N+1)
  # logits : (B, n_classes)
  # mode   : "mask"(이미지에 mask 곱하기, 논문 Fig.14 스타일) | "heatmap"(jet 오버레이)

  B, C, H, W = images.shape
  n = min(B, max_images)

  images = images.detach().cpu().float()
  if mean is not None and std is not None:
    m = torch.tensor(mean).view(1, -1, 1, 1)
    s = torch.tensor(std).view(1, -1, 1, 1)
    images = images * s + m
  imgs = images.clamp(0, 1).permute(0, 2, 3, 1).numpy()      # (B, H, W, C)
  if C == 1:
    imgs = np.repeat(imgs, 3, axis=-1)

  with torch.no_grad():
    rollout = attention_rollout(scores.detach())             # (B, 1+N, 1+N)
  mask = rollout[:, 0, 1:]                                   # (B, N)
  L = math.isqrt(mask.size(-1))
  assert L * L == mask.size(-1), "Num patches is not perfect square"

  mask = mask.reshape(B, 1, L, L)
  mask = F.interpolate(mask, size=(H, W), mode="bilinear", align_corners=False)  # (B,1,H,W)

  # 이미지 정규화
  mn = mask.amin(dim=(1, 2, 3), keepdim=True)
  mx = mask.amax(dim=(1, 2, 3), keepdim=True)
  mask = ((mask - mn) / (mx - mn + 1e-8)).squeeze(1).cpu().numpy()   # (B, H, W)

  probs = torch.softmax(logits.detach().cpu().float(), dim=-1)
  conf, preds = probs.max(dim=-1)
  targets = targets.detach().cpu()
  name = (lambda i: class_names[int(i)]) if class_names is not None else (lambda i: str(int(i)))

  fig, axes = plt.subplots(n, 2, figsize=(6, 3 * n), squeeze=False)
  cmap = plt.get_cmap("jet")

  for i in range(n):
    if mode == "mask":
      overlay = imgs[i] * mask[i][..., None]
    else:
      heat = cmap(mask[i])[..., :3]
      overlay = 0.5 * imgs[i] + 0.5 * heat

    axes[i, 0].imshow(imgs[i])
    axes[i, 0].set_title(f"Target: {name(targets[i])}", fontsize=10)

    correct = int(preds[i]) == int(targets[i])
    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title(f"Pred: {name(preds[i])} ({conf[i]:.0%})",
                         fontsize=10, color="green" if correct else "red")

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

  plt.tight_layout()
  return fig

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/methods.py


# Smoke Test

In [21]:
from dataclasses import dataclass

@dataclass
class ViTConfig:
  img_size: int=224
  patch_size: int=16
  in_channels: int=3
  n_classes: int =1000
  d_model: int =768
  n_layers: int =12
  heads: int =12
  bias: bool =True
  pos_p: float=0.1
  attn_p: float=0.1
  proj_p: float=0.1

In [22]:
from model import *
from methods import *

In [23]:
cfg = ViTConfig()
net = VisionTransformer(cfg)
print(net)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_embed): Embedding(197, 768)
  (pos_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (out_fc): Linear(in_features=768, out_features=768, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplac

In [24]:
import torch
net.eval()
x = torch.randn(10, 3, 224, 224)
with torch.no_grad():
    slow, scores = net(x, return_attn=True)

KeyboardInterrupt: 

In [ ]:
print(slow)
print(scores[0][0])

In [ ]:
import matplotlib.pyplot as plt




dist = compute_mean_attention_dist(scores, patch_size=cfg.patch_size)
plot_attention_distance(dist, title="ViT (absolute pos embed)")
plt.show()

In [ ]:
n_layers = 12
B = 2
heads = 12
N = 14 * 14

scores = torch.softmax(
    torch.randn(n_layers, B, heads, N + 1, N + 1), dim=-1
)
dist = compute_mean_attention_dist(scores, patch_size=16)
print(dist.min(), dist.max())

print("Output shape:", dist.shape)
print("Mean distance:", dist.mean().item())
print("Min distance:", dist.min().item())
print("Max distance:", dist.max().item())

In [ ]:
patch_scores = scores[..., 1:, 1:]

print("Attention min:", patch_scores.min().item())
print("Attention max:", patch_scores.max().item())
print("Attention mean:", patch_scores.mean().item())